# 核对 Codex 的修框依据

这批样本来自 nuScenes val 的 CenterPoint→GT 原版回归轨迹。每条记录对应一步，照片和点云中的框均为该步执行前的框。动作和档位来自轨迹，Codex 按画面解释该步能得到多强的视觉支持。

在数据目录打开本 Notebook，选择 `qwen-vl-sft` 内核。先运行「载入数据」，再调整「挑选样本」单元的筛选项。每次修改后只需重新运行该单元。最下方可以保存你自己的判定。

## 载入数据

In [ ]:
from pathlib import Path
import base64
import html
import json
import math
import os
import fcntl
from collections import Counter
from datetime import datetime, timezone
from IPython.display import HTML, display

# 在其他目录启动内核时，改为本次数据集的绝对路径。
ROOT = Path.cwd().resolve()

def read_index(name):
    result = {}
    with (ROOT / name).open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row["id"] in result:
                raise ValueError(f"{name} 出现重复样本 {row['id']}")
            result[row["id"]] = row
    return result

tasks = read_index("tasks.jsonl")
observations = read_index("observations.jsonl")
labels = read_index("labels.jsonl")
if set(tasks) != set(observations) or set(tasks) != set(labels):
    raise ValueError("请核对 tasks.jsonl、observations.jsonl 和 labels.jsonl 的样本 ID。")

VIEWS = ["LIDAR", "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
         "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT"]
VIEW_ZH = ["LiDAR 局部视图", "左前", "正前", "右前", "左后", "正后", "右后"]
SUPPORT = {"supported":"支持", "partial":"部分支持", "insufficient":"证据不足", "contradicted":"可见证据冲突"}
AXIS = {"x":"参考 LiDAR x", "y":"参考 LiDAR y", "z":"参考 LiDAR z",
        "length":"框长", "width":"框宽", "height":"框高", "yaw":"绕参考 LiDAR z 的 yaw"}
REVIEW_FILE = ROOT / "manual_reviews.jsonl"

def human_action(action):
    if action["type"] == "stop":
        return "STOP · 停止编辑"
    axis, v = action["axis"], action["value"]
    amount = f"{math.degrees(v):+.2f}°" if axis == "yaw" else f"{v * 100:+.2f} cm"
    return f"{AXIS[axis]} {amount}"

def read_reviews():
    if not REVIEW_FILE.is_file():
        return {}
    with REVIEW_FILE.open(encoding="utf-8") as f:
        fcntl.flock(f, fcntl.LOCK_SH)
        return {r["id"]:r for r in map(json.loads, f)}

def show_images(observation):
    blocks = []
    for view, name, rel in zip(VIEWS, VIEW_ZH, observation["images"]):
        content = base64.b64encode((ROOT / rel).read_bytes()).decode("ascii")
        marker = "" if view == "LIDAR" else (" · 有框投影" if view in observation["projected"] else " · 无框投影")
        caption = html.escape(f"{name}  {view}{marker}")
        span = "grid-column:1/-1;" if view == "LIDAR" else ""
        blocks.append(f'<figure style="margin:0;{span}"><figcaption>{caption}</figcaption>'
                      f'<img alt="{caption}" style="max-width:100%;height:auto" src="data:image/jpeg;base64,{content}"></figure>')
    display(HTML('<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,260px),1fr));gap:12px">'
                 + "".join(blocks) + '</div>'))

def brief_counts():
    print(f"{len(labels)} 个动作，已手动核对 {len(read_reviews())} 个。")
    counts = Counter(r["assessment"]["support"] for r in labels.values())
    print("教师证据分布", {SUPPORT[k]:v for k,v in counts.items()})

brief_counts()

## 挑选样本

`SAMPLE_ID` 有值时可直接定位样本。留空时，`INDEX` 表示筛选后第几个样本，从 0 开始。`SHOW_TEACHER = False` 可隐藏教师说明；同时把 `SHOW_ACTION` 设为 `False`，就能先看执行前观测，独立判断下一步。动作按参考 LiDAR 的轴定义，屏幕左右随相机视角变化。

In [ ]:
SAMPLE_ID = ""               # 例如 "v0001"
AXIS_FILTER = ""             # x / y / z / length / width / height / yaw / STOP
SUPPORT_FILTER = ""          # supported / partial / insufficient / contradicted
CATEGORY_FILTER = ""         # 例如 car 或 traffic_cone
ONLY_UNREVIEWED = False
INDEX = 0
SHOW_TEACHER = True
SHOW_ACTION = True

# 直接定位时优先使用 SAMPLE_ID。
reviews = read_reviews()
if SAMPLE_ID:
    if SAMPLE_ID not in tasks:
        raise KeyError(f"没有这个样本 ID: {SAMPLE_ID}")
    matches = [SAMPLE_ID]
else:
    matches = []
    for key, task in tasks.items():
        a = task["action"]
        axis = "STOP" if a["type"] == "stop" else a["axis"]
        if AXIS_FILTER and axis.lower() != AXIS_FILTER.lower():
            continue
        if SUPPORT_FILTER and labels[key]["assessment"]["support"] != SUPPORT_FILTER:
            continue
        if CATEGORY_FILTER and task["category"] != CATEGORY_FILTER:
            continue
        if ONLY_UNREVIEWED and key in reviews:
            continue
        matches.append(key)

selected_id = None
if not matches:
    print("筛选结果为空，可放宽筛选条件后重跑本单元。")
else:
    pos = 0 if SAMPLE_ID else INDEX
    if not (0 <= pos < len(matches)):
        raise IndexError(f"筛选后有 {len(matches)} 个样本，INDEX 的范围为 0 到 {len(matches)-1}。")
    selected_id = matches[pos]
    t, o, record = tasks[selected_id], observations[selected_id], labels[selected_id]
    a = t["action"]
    print(f"{selected_id} · {t.get('scene_name', t.get('scene_token',''))} · {t['category']} · 筛选后第 {pos+1}/{len(matches)} 条")
    print(f"轨迹 {t['episode_id']}  |  执行前状态 {t['state_id']}  |  动作 {t.get('action_id','')}  |  源文件行 {t.get('source_line','')}")
    if SHOW_ACTION:
        print("轨迹动作", human_action(a), a)
    print("执行前框 [x, y, z, 长, 宽, 高, yaw弧度]", t["box"])
    show_images(o)
    if SHOW_TEACHER:
        written = record["assessment"]
        print("Codex 证据等级", SUPPORT[written["support"]], f"({written['support']})")
        print("LiDAR", written["lidar_evidence"])
        for fact in written["camera_evidence"]:
            print(fact["view"], fact["observation"])
        if not written["camera_evidence"]:
            print("相机：教师未引用可辨认的带框视角。")
        print("动作依据", written["rationale"])
    if selected_id in reviews:
        last = reviews[selected_id]
        print("你的最近一次判定", last["verdict"], last["note"], last["utc"])
    else:
        print("人工复核：待核对")

## 保存人工判定

看完当前样本后，把下面的 `VERDICT` 改成 `通过`、`需修改` 或 `不确定`，填写备注后运行。结果追加到同目录的 `manual_reviews.jsonl`。同一样本可以更新判定，浏览时显示最近的一次，历史判断仍可回查。

In [ ]:
VERDICT = ""               # 通过 / 需修改 / 不确定；留空时仅查看
NOTE = ""

if VERDICT:
    if selected_id is None:
        raise ValueError("请先在上一单元选定样本。")
    if VERDICT not in ("通过", "需修改", "不确定"):
        raise ValueError("VERDICT 应选 通过、需修改 或 不确定。")
    with REVIEW_FILE.open("a+", encoding="utf-8") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.seek(0)
        latest = None
        for old in map(json.loads, f):
            if old["id"] == selected_id:
                latest = old
        if latest and (latest["verdict"], latest["note"]) == (VERDICT, NOTE.strip()):
            print("已有相同判定", selected_id)
        else:
            decision = {"id":selected_id, "episode_id":tasks[selected_id]["episode_id"],
                        "state_id":tasks[selected_id]["state_id"], "verdict":VERDICT, "note":NOTE.strip(),
                        "utc":datetime.now(timezone.utc).isoformat(timespec="seconds")}
            f.seek(0, os.SEEK_END)
            print(json.dumps(decision, ensure_ascii=False), file=f)
            f.flush()
            os.fsync(f.fileno())
            print("已保存", selected_id, VERDICT)
else:
    print("填写 VERDICT 后运行本单元即可保存。")